In [ ]:
!pip install transformers torch scikit-learn -q

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report
from sklearn.preprocessing import LabelEncoder

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

df = pd.read_csv('combined_en.csv')
print(df['label'].value_counts())

In [ ]:
le = LabelEncoder()
df['label_id'] = le.fit_transform(df['label'])

print(f'Classes: {le.classes_}')
print('\nMapping:')
for i, cls in enumerate(le.classes_):
    print(f'{i} -> {cls}')

X_train, X_test, y_train, y_test = train_test_split(
    df['text'].values,
    df['label_id'].values,
    test_size=0.2,
    random_state=42,
    stratify=df['label_id'].values
)

print(f'\nTrain: {len(X_train)}')
print(f'Test: {len(X_test)}')

In [ ]:
tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')

class TextDataset(Dataset):

    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]

        encoding = self.tokenizer(
            text,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'label': torch.tensor(label, dtype=torch.long)
        }

train_dataset = TextDataset(X_train, y_train, tokenizer)
test_dataset = TextDataset(X_test, y_test, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

print(f'Train batch: {len(train_loader)}\nTest batch: {len(test_loader)}')

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=5,
    ignore_mismatched_sizes=True
)

model = model.to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'Total params: {total_params:,}')
print(f'Trainable params: {trainable_params:,}')
print(f'Model Device: {next(model.parameters()).device}');

In [ ]:
EPOCHS = 6
optimizer = AdamW(model.parameters(), lr=2e-5)

total_steps = len(train_loader) * EPOCHS

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=50,
    num_training_steps=total_steps
)

print(f'Epochs: {EPOCHS}\nBatches/epoch: {len(train_loader)}\nTotal steps: {total_steps}')

In [ ]:
def train_epoch(model, loader, optimizer, scheduler, device):
    model.train()

    total_loss = 0
    correct = 0
    total = 0

    for batch in loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)

        # clean grad
        optimizer.zero_grad()

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        logits = outputs.logits

        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()
        scheduler.step()

        total_loss += loss.item()

        preds = torch.argmax(logits, dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    avg_loss = total_loss / len(loader)
    accuracy = correct / total
    return avg_loss, accuracy

def eval_epoch(model, loader, device):
    model.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds = torch.argmax(outputs.logits, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    f1 = f1_score(all_labels, all_preds, average='macro')

    return f1, all_preds, all_labels

In [ ]:
import time
import copy

best_f1 = 0
best_model_state = None

torch.cuda.empty_cache()

for epoch in range(EPOCHS):
    start_time = time.time()

    train_loss, train_acc = train_epoch(model, train_loader, optimizer, scheduler, device)
    f1, preds, labels_true = eval_epoch(model, test_loader, device)
    elapsed = time.time() - start_time

    print(f'Epoch: {epoch+1}/{EPOCHS} | '
          f'Loss: {train_loss:.4f} | '
          f'Train Acc: {train_acc:.4f} | '
          f'F1: {f1:.4f} | '
          f'Time: {elapsed:.0f}s')

    if f1 > best_f1:
        best_f1 = f1
        best_model_state = copy.deepcopy(model.state_dict())
        print(f'F1={best_f1:.4f}')

print(f'\nBest F1: {best_f1:.4f}')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import torch
import os
os.makedirs('/content/drive/MyDrive/psycholinguistic_detector', exist_ok=True)

torch.save(best_model_state, '/content/drive/MyDrive/psycholinguistic_detector/distilbert_best.pt')
tokenizer.save_pretrained('/content/drive/MyDrive/psycholinguistic_detector/tokenizer')

import json
classes = le.classes_.tolist()
with open('/content/drive/MyDrive/psycholinguistic_detector/classes.json', 'w') as f:
  json.dump(classes, f)

print(f'Best F1: {best_f1:.4f}')
print(f'Classes: {classes}')